# Day 175 — Ollama on Colab: Local LLM Inference
## Month 10 | Google Colab (T4 GPU)

---

### Month 10 Scorecard
| Day | Topic | Score |
|-----|-------|-------|
| 169 | LangChain Chains & Memory | ✅ 80/80+10★ |
| 170 | LangChain Tools & Agents | ✅ 80/80+10★ |
| 171 | Document Loaders + LCEL | ✅ 80/80+10★ |
| 172 | LangChain Capstone | ✅ 90/90+10★ |
| 173 | MLflow Experiment Tracking | ✅ 90/90+10★ |
| 174 | MLflow Model Registry + Run Comparison | ✅ 90/90+10★ |
| **175** | **Ollama on Colab — Local LLM Inference** | **← Today** |

**Running Total: 520/520+60★ | All 6 days PERFECT**

---

### Today's Scorecard
| Task | Topic | Points |
|------|-------|--------|
| T1 | Install Ollama + Start Server + Health Check | 15 |
| T2 | Pull TinyLlama + Verify Model Available | 15 |
| T3 | Raw REST API Inference (requests) | 20 |
| T4 | LangChain + Ollama Integration (chain) | 20 |
| T5 | ReviewPulse Batch Inference + NRA Insight | 20 |
| ★  | Groq vs Ollama Latency Comparison Table | 10★ |
| **Total** | | **90/90 + 10★** |

**Dataset:** ReviewPulse India (600 rows, seed=155)
**Model:** TinyLlama (637 MB) via Ollama on Colab T4
**Why Colab:** Local HP laptop (8GB RAM, MX130) cannot run Ollama — Colab T4 gives 16GB VRAM

---

> ⚠️ **Runtime Note:** After running Cell 1 (install), do NOT restart runtime.
> Ollama server runs as a background subprocess — all cells must run in sequence in one session.


---
## Section 1: Concept Notes

### What is Ollama?
**Ollama** is a tool that packages open-source LLMs (TinyLlama, Phi-3, Llama 3, Mistral, etc.)
with a local HTTP server exposing an OpenAI-compatible REST API at `http://localhost:11434`.

```
┌─────────────────────────────────────────────────────────────┐
│                      Ollama Architecture                     │
│                                                             │
│  ollama serve  ──▶  HTTP Server (port 11434)               │
│                         │                                   │
│              ┌──────────┴──────────┐                        │
│         /api/generate          /api/chat                    │
│              │                      │                       │
│      requests.post()        LangChain Ollama()             │
│              │                      │                       │
│         Raw JSON              LangChain Chain               │
└─────────────────────────────────────────────────────────────┘
```

### Why use Ollama instead of Groq?
| Dimension | Ollama (local) | Groq (cloud API) |
|---|---|---|
| **Cost** | Free forever | Free tier has rate limits |
| **Privacy** | Data never leaves machine | Data sent to cloud |
| **Speed** | Depends on hardware | Very fast (LPU chips) |
| **Model size** | Limited by RAM/VRAM | Any model available |
| **Offline** | ✅ Works offline | ❌ Needs internet |
| **Use case** | Sensitive data, dev testing | Production, large models |

### Colab Strategy (Your Hardware Constraint)
- **Local laptop:** 8GB RAM, MX130 2GB VRAM → TinyLlama would OOM
- **Colab T4:** 12GB RAM + 16GB VRAM → runs TinyLlama comfortably
- **Colab workflow:** Install → start subprocess → pull model → use API

### TinyLlama Model Facts
- **Size:** ~637 MB (Q4_0 quantized)
- **Parameters:** 1.1B
- **Context window:** 2048 tokens
- **Best for:** Fast inference, testing pipelines, low-resource dev
- **Not best for:** Long reasoning, complex instructions

### LangChain + Ollama (pinned versions)
```python
# With langchain-community==0.2.16
from langchain_community.llms import Ollama
llm = Ollama(model="tinyllama", temperature=0.1)
```

### REST API Schema
```python
POST http://localhost:11434/api/generate
{
  "model": "tinyllama",
  "prompt": "Your prompt here",
  "stream": False,        # ← False = wait for full response
  "options": {
    "temperature": 0.1,
    "num_predict": 200   # max tokens
  }
}
# Response key: response["response"]
```


---
## Section 2: Raw Data (DO NOT MODIFY)

In [1]:
# ── RAW DATA GENERATION ─────────────────────────────────────────────────────
# ReviewPulse India | 600 rows | seed=155
# DO NOT modify this cell or the df_raw variable

import pandas as pd
import numpy as np

np.random.seed(155)
n = 600

products   = ['Smartphone', 'Laptop', 'Headphones', 'Tablet', 'Smartwatch']
sentiments = ['positive', 'negative', 'neutral']

df_raw = pd.DataFrame({
    'review_id'        : range(1, n + 1),
    'product_category' : np.random.choice(products, n),
    'rating'           : np.random.choice([1, 2, 3, 4, 5], n,
                             p=[0.10, 0.15, 0.20, 0.30, 0.25]),
    'sentiment'        : np.random.choice(sentiments, n,
                             p=[0.55, 0.25, 0.20]),
    'review_length'    : np.random.randint(20, 500, n),
    'verified_purchase': np.random.choice([0, 1], n, p=[0.30, 0.70]),
    'helpful_votes'    : np.random.randint(0, 50, n),
})

# Derived target (same as all Month 10 days)
df_raw['high_rating'] = (df_raw['rating'] >= 4).astype(int)

print("✅ Raw data loaded")
print(f"Shape: {df_raw.shape}")
print(f"\nClass distribution (high_rating):")
print(df_raw['high_rating'].value_counts())
print(f"\nSample rows:")
print(df_raw.head(3).to_string())


✅ Raw data loaded
Shape: (600, 8)

Class distribution (high_rating):
high_rating
1    316
0    284
Name: count, dtype: int64

Sample rows:
   review_id product_category  rating sentiment  review_length  verified_purchase  helpful_votes  high_rating
0          1       Smartwatch       4  negative            270                  1             20            1
1          2           Laptop       4  positive            137                  1              6            1
2          3       Smartphone       5  positive             27                  1             20            1


---
## Section 3: Practice Tasks

### T1: Install Ollama + Start Server + Health Check (15 pts)

In [2]:
# ── T1a: INSTALL OLLAMA (robust) ─────────────────────────────────────────────────────
# Goal: Download and install Ollama binary on Colab, with fallback if script fails.
# Expected output: "ollama version X.X.X" printed successfully.

import subprocess
import os
import time

# 1. Install zstd (required for model decompression)
subprocess.run(['sudo', 'apt', 'update'], check=False)
subprocess.run(['sudo', 'apt', 'install', '-y', 'zstd'], check=False)

# 2. Run official install script with -y
print("Installing Ollama via official script...")
install_result = subprocess.run(
    ['bash', '-c', 'curl -fsSL https://ollama.com/install.sh | sh -s -- -y'],
    capture_output=True,
    text=True
)

# 3. If script failed, fallback: download binary from GitHub
if install_result.returncode != 0:
    print("Official script failed. Falling back to manual binary download...")
    # Download the latest Linux amd64 binary (replace with your arch if needed)
    subprocess.run(['sudo', 'curl', '-L',
                    'https://github.com/ollama/ollama/releases/latest/download/ollama-linux-amd64',
                    '-o', '/usr/local/bin/ollama'], check=False)
    subprocess.run(['sudo', 'chmod', '+x', '/usr/local/bin/ollama'], check=False)

# 4. Ensure /usr/local/bin is in PATH (Colab sometimes doesn't include it)
os.environ['PATH'] += ':/usr/local/bin'

# 5. Verify binary exists and print version
try:
    version_result = subprocess.run(['ollama', '--version'], capture_output=True, text=True, timeout=10)
    print("\n✅ Ollama version:", version_result.stdout.strip())
except FileNotFoundError:
    print("❌ Still could not find 'ollama'. Please check internet connection and retry.")
    # Additional diagnostic: list /usr/local/bin
    print("Contents of /usr/local/bin:")
    subprocess.run(['ls', '-la', '/usr/local/bin'])

Installing Ollama via official script...

✅ Ollama version: Warning: could not connect to a running Ollama instance


In [3]:
# ── T1b: START OLLAMA SERVER AS BACKGROUND SUBPROCESS ───────────────────────
# Goal: Start 'ollama serve' in background so API is available at localhost:11434
# Method: subprocess.Popen (non-blocking) + time.sleep for startup

import subprocess
import time

# Start server in background
server_proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
print(f"Server PID: {server_proc.pid}")
print("Waiting 8 seconds for server to initialize...")
time.sleep(8)
print("✅ Server startup wait complete")


Server PID: 6139
Waiting 8 seconds for server to initialize...
✅ Server startup wait complete


In [4]:
# ── T1c: HEALTH CHECK ───────────────────────────────────────────────────────
# Goal: Confirm Ollama server is running via HTTP request
# Expected: HTTP 200 or response body containing "Ollama is running"

import requests

try:
    resp = requests.get('http://localhost:11434', timeout=10)
    print(f"Status code : {resp.status_code}")
    print(f"Response    : {resp.text[:200]}")
    if resp.status_code == 200:
        print("\n✅ T1 COMPLETE — Ollama server is running")
    else:
        print("\n⚠️  Unexpected status code")
except Exception as e:
    print(f"❌ Health check failed: {e}")
    print("Try increasing sleep time in T1b and re-run from T1b")


Status code : 200
Response    : Ollama is running

✅ T1 COMPLETE — Ollama server is running


---
### T2: Pull TinyLlama + Verify Model Available (15 pts)

In [5]:
# ── T2a: PULL TINYLLAMA MODEL ───────────────────────────────────────────────
# Goal: Download TinyLlama (637 MB) to Colab — takes ~1-2 min on Colab
# Why TinyLlama: Smallest viable model, fits in Colab RAM, fast inference
# Note: 'stream' in subprocess shows live download progress

import subprocess

print("Pulling TinyLlama — this takes 1–3 minutes on Colab...")
print("(You will see download progress in stderr)")

pull_result = subprocess.run(
    ['ollama', 'pull', 'tinyllama'],
    capture_output=False,   # show live output
    text=True,
    timeout=300             # 5-minute timeout
)
print(f"\nReturn code: {pull_result.returncode}")
if pull_result.returncode == 0:
    print("✅ TinyLlama pulled successfully")
else:
    print("❌ Pull failed — check internet connection or try again")


Pulling TinyLlama — this takes 1–3 minutes on Colab...
(You will see download progress in stderr)

Return code: 0
✅ TinyLlama pulled successfully


In [6]:
# ── T2b: VERIFY MODEL IS LISTED ─────────────────────────────────────────────
# Goal: Confirm tinyllama appears in 'ollama list' output
# Why this matters: Confirms model is stored and ready for inference

import subprocess

list_result = subprocess.run(['ollama', 'list'], capture_output=True, text=True)
print("Available models:")
print(list_result.stdout)

# Also verify via API
import requests
models_resp = requests.get('http://localhost:11434/api/tags')
models_json = models_resp.json()

print("\nModels via API:")
for m in models_json.get('models', []):
    size_gb = m.get('size', 0) / 1e9
    print(f"  {m['name']:<30} {size_gb:.2f} GB")

# Extract and print tinyllama size for NRA later
tinyllama_info = [m for m in models_json.get('models', []) if 'tinyllama' in m['name']]
if tinyllama_info:
    size_mb = tinyllama_info[0]['size'] / 1e6
    print(f"\n✅ T2 COMPLETE — TinyLlama size: {size_mb:.1f} MB")
    TINYLLAMA_SIZE_MB = size_mb
else:
    print("⚠️  TinyLlama not found in API list — re-pull may be needed")
    TINYLLAMA_SIZE_MB = 637.0  # fallback


Available models:
NAME                ID              SIZE      MODIFIED               
tinyllama:latest    2644915ede35    637 MB    Less than a second ago    


Models via API:
  tinyllama:latest               0.64 GB

✅ T2 COMPLETE — TinyLlama size: 637.7 MB


---
### T3: Raw REST API Inference (20 pts)

In [7]:
# ── T3a: FIRST INFERENCE VIA REST API ───────────────────────────────────────
# Goal: Send a ReviewPulse-themed prompt to Ollama API using requests.post()
# Method: POST to /api/generate with stream=False
# Lock in: response_time_s, response text — needed for NRA + bonus task

import requests
import time

# Build a business-relevant prompt using ReviewPulse context
prompt_text = (
    "You are a product analytics assistant. "
    "A Smartphone product has 45% positive reviews, 30% negative reviews, and 25% neutral reviews. "
    "The average rating is 3.6 out of 5. "
    "In 2 sentences, what is the most important action the product team should take?"
)

payload = {
    "model"  : "tinyllama",
    "prompt" : prompt_text,
    "stream" : False,
    "options": {
        "temperature" : 0.1,
        "num_predict" : 150
    }
}

print("Sending request to Ollama REST API...")
t0 = time.time()
resp = requests.post('http://localhost:11434/api/generate', json=payload, timeout=120)
OLLAMA_LATENCY_S = round(time.time() - t0, 2)

if resp.status_code == 200:
    result = resp.json()
    print(f"\n✅ Status     : {resp.status_code}")
    print(f"⏱  Latency    : {OLLAMA_LATENCY_S}s")
    print(f"🔢 Eval tokens: {result.get('eval_count', 'N/A')}")
    print(f"\n📝 Response:")
    print(result['response'])
    OLLAMA_RESPONSE_1 = result['response']
else:
    print(f"❌ Error: {resp.status_code} — {resp.text}")
    OLLAMA_LATENCY_S = None
    OLLAMA_RESPONSE_1 = None


Sending request to Ollama REST API...

✅ Status     : 200
⏱  Latency    : 3.17s
🔢 Eval tokens: 150

📝 Response:
The product team should take the following action based on the Smartphone product's 45% positive reviews, 30% negative reviews, and 25% neutral reviews:

1. Analyze the data: The team should analyze the data to identify which reviews are most influential in shaping consumer perception of the product. This information can help them make informed decisions about how to improve the product's overall rating and reputation.

2. Address negative reviews: If the team identifies a significant number of negative reviews, they should take steps to address these issues. For example, they could offer refunds or replacements for defective products, provide additional customer support, or improve the


In [8]:
# ── T3b: PARSE API RESPONSE FIELDS ─────────────────────────────────────────
# Goal: Extract and display all key fields from the Ollama API response
# These fields are important for monitoring, latency benchmarking, and debugging

import requests
import json

# Re-run a short prompt to inspect response structure
test_payload = {
    "model"  : "tinyllama",
    "prompt" : "What is sentiment analysis? Answer in one sentence.",
    "stream" : False,
    "options": {"temperature": 0.0, "num_predict": 60}
}

resp = requests.post('http://localhost:11434/api/generate', json=test_payload, timeout=60)
r = resp.json()

print("=== Ollama API Response Fields ===")
print(f"model          : {r.get('model')}")
print(f"response       : {r.get('response', '')[:120]}")
print(f"done           : {r.get('done')}")
print(f"total_duration : {r.get('total_duration', 0) / 1e9:.3f}s  (nanoseconds → seconds)")
print(f"load_duration  : {r.get('load_duration', 0) / 1e9:.3f}s")
print(f"eval_count     : {r.get('eval_count')} tokens")
print(f"eval_duration  : {r.get('eval_duration', 0) / 1e9:.3f}s")

if r.get('eval_count') and r.get('eval_duration'):
    tokens_per_sec = r['eval_count'] / (r['eval_duration'] / 1e9)
    print(f"tokens/sec     : {tokens_per_sec:.1f}")
    TOKENS_PER_SEC = round(tokens_per_sec, 1)
else:
    TOKENS_PER_SEC = None

print("\n✅ T3 COMPLETE — REST API inference verified")


=== Ollama API Response Fields ===
model          : tinyllama
response       : Sentiment analysis is the process of identifying and interpreting the emotional or positive-negative nature of text, suc
done           : True
total_duration : 0.388s  (nanoseconds → seconds)
load_duration  : 0.104s
eval_count     : 60 tokens
eval_duration  : 0.268s
tokens/sec     : 224.1

✅ T3 COMPLETE — REST API inference verified


---
### T4: LangChain + Ollama Integration (20 pts)

In [9]:
# ── T4a: INSTALL / VERIFY LANGCHAIN (pinned versions) ───────────────────────
# Month 10 pinned versions — NEVER change these
# LangChain 0.3+ removed Ollama from core — must use langchain_community

!pip install -q langchain==0.2.16 langchain-community==0.2.16

# Restart reminder: NOT needed here (Ollama doesn't affect Python packages)
# If you get import errors, check that these exact versions are installed

import langchain
import langchain_community
print(f"langchain version           : {langchain.__version__}")
print(f"langchain_community version : {langchain_community.__version__}")


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-groq 1.1.3 requires langchain-core<2.0.0,>=1.4.0, but you have langchain-core 0.2.43 which is incompatible.
langgraph 1.2.5 requires langchain-core<2,>=1.4.7, but you have langchain-core 0.2.43 which is incompatible.
langgraph-prebuilt 1.1.0 requires langchain-core>=1.3.1, but you have langchain-core 0.2.43 which is incompatible.
langgraph-sdk 0.4.2 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.2.43 which is incompatible.
langchain version           : 0.2.16
langchain_community version : 0.2.16


In [10]:
# ── T4b: INSTANTIATE OLLAMA LLM VIA LANGCHAIN ───────────────────────────────
# Goal: Create an Ollama LLM object and run a basic .invoke() call
# Import path: langchain_community.llms (NOT langchain.llms in 0.2.x)

from langchain_community.llms import Ollama
from langchain.prompts import PromptTemplate

# Instantiate — points to local Ollama server
llm_ollama = Ollama(
    model       = "tinyllama",
    temperature = 0.1,
    num_predict = 150,
)

# Quick sanity check
print("Invoking LangChain Ollama LLM...")
test_response = llm_ollama.invoke("What is a product review sentiment in one sentence?")
print(f"\n✅ LangChain Ollama response:")
print(test_response)


Invoking LangChain Ollama LLM...

✅ LangChain Ollama response:
A product review sentiment in one sentence is a summary of the overall tone and sentiment expressed by the reviewer, typically expressing their thoughts on whether they would recommend or not recommend the product. It can also include specific details about the product's features, benefits, drawbacks, and overall value for money.


In [11]:
# ── T4c: BUILD PROMPT TEMPLATE + CHAIN (LCEL) ───────────────────────────────
# Goal: Create a PromptTemplate → Ollama chain using LCEL pipe operator.
# Method: Define a template with variables, then chain: prompt | llm | StrOutputParser.

from langchain.prompts import PromptTemplate
from langchain.schema.output_parser import StrOutputParser

# Prompt template for ReviewPulse product analysis – using triple quotes for multiline
review_prompt = PromptTemplate(
    input_variables=["product", "avg_rating", "positive_pct", "negative_pct"],
    template="""You are a data analyst reviewing customer feedback for an e-commerce platform.
Product category: {product}
Average rating: {avg_rating}/5
Positive reviews: {positive_pct}%
Negative reviews: {negative_pct}%

Provide exactly ONE specific action the product team should take. Start your response with the word 'Action:'"""
)

# LCEL chain
parser = StrOutputParser()
review_chain = review_prompt | llm_ollama | parser

print("✅ LCEL chain built: review_prompt | llm_ollama | parser")
print(f"Chain type: {type(review_chain)}")

✅ LCEL chain built: review_prompt | llm_ollama | parser
Chain type: <class 'langchain_core.runnables.base.RunnableSequence'>


In [12]:
# ── T4d: RUN CHAIN ON REVIEWPULSE AGGREGATED STATS ─────────────────────────
# Goal: Compute real stats from df_raw, feed into chain, get LLM insight
# Number values must come from printed cell output (NRA rule)

import time

# Compute real stats per product category
df_work = df_raw.copy()
df_work['sentiment_bin'] = df_work['sentiment'].map(
    {'positive': 'positive', 'negative': 'negative', 'neutral': 'neutral'}
)

# Get stats for Smartphone (most reviews)
for product in ['Smartphone', 'Laptop', 'Headphones']:
    subset = df_work[df_work['product_category'] == product]
    avg_r   = round(subset['rating'].mean(), 2)
    pos_pct = round((subset['sentiment'] == 'positive').mean() * 100, 1)
    neg_pct = round((subset['sentiment'] == 'negative').mean() * 100, 1)

    print(f"\n{'='*60}")
    print(f"Product: {product} | n={len(subset)} | avg_rating={avg_r} | pos={pos_pct}% | neg={neg_pct}%")
    print(f"{'='*60}")

    t0 = time.time()
    response = review_chain.invoke({
        "product"      : product,
        "avg_rating"   : avg_r,
        "positive_pct" : pos_pct,
        "negative_pct" : neg_pct
    })
    latency = round(time.time() - t0, 2)

    print(f"LLM Insight (latency: {latency}s):")
    print(response)

print("\n✅ T4 COMPLETE — LangChain Ollama chain executed on ReviewPulse data")



Product: Smartphone | n=108 | avg_rating=3.21 | pos=53.7% | neg=31.5%
LLM Insight (latency: 0.77s):
Action: Based on the positive feedback received, the product team should prioritize improving the user experience by adding more customization options for users to personalize their devices' settings. This will not only enhance the overall user experience but also increase customer satisfaction and loyalty. The team could consider implementing a feature that allows users to choose from different color schemes or themes, as well as adjusting the brightness level and sound volume levels. By providing more customization options, the product team can ensure that customers feel like their devices are tailored to their individual preferences, leading to increased engagement and loyalty.

Product: Laptop | n=129 | avg_rating=3.42 | pos=58.1% | neg=18.6%
LLM Insight (latency: 0.66s):
Action: To improve the Averaage rating for this product category, the product team should focus on improving the

---
### T5: ReviewPulse Batch Inference + NRA Insight (20 pts)

In [13]:
# ── T5a: BATCH INFERENCE ON SAMPLE REVIEWS ──────────────────────────────────
# Goal: Run Ollama inference on 5 sample reviews and compare LLM classification
#       to the actual sentiment column.
# Method: Build a classification chain that uses rating to determine sentiment,
#         run on 5 reviews, compute match rate and average latency.

from langchain.prompts import PromptTemplate
from langchain.schema.output_parser import StrOutputParser
import time

classify_prompt = PromptTemplate(
    input_variables=["review_id", "product", "rating", "review_length"],
    template=(
        "A customer left a review for a {product}. "
        "Rating: {rating}/5. Review length: {review_length} words. "
        "Based ONLY on the rating (1-2=negative, 3=neutral, 4-5=positive), "
        "respond with exactly one word: positive, negative, or neutral."
    )
)
classify_chain = classify_prompt | llm_ollama | parser   # parser defined earlier

# Sample 5 reviews — one from each rating level
sample = df_raw.groupby('rating').first().reset_index()
sample = sample[['review_id', 'product_category', 'rating', 'review_length', 'sentiment']].head(5)

print("Batch inference on 5 reviews...\n")   # FIXED: no newline inside string

results = []
for _, row in sample.iterrows():
    t0 = time.time()
    llm_pred = classify_chain.invoke({
        "review_id": row['review_id'],
        "product": row['product_category'],
        "rating": row['rating'],
        "review_length": row['review_length']
    })
    latency = round(time.time() - t0, 2)
    llm_clean = llm_pred.strip().lower().split()[0] if llm_pred.strip() else "unknown"
    match = "✅" if llm_clean == row['sentiment'] else "❌"
    results.append({
        'review_id': row['review_id'],
        'rating': row['rating'],
        'actual': row['sentiment'],
        'llm_pred': llm_clean,
        'match': match,
        'latency_s': latency
    })
    print(f"review_id={row['review_id']} | rating={row['rating']} | actual={row['sentiment']:<9} | llm={llm_clean:<9} | {match} | {latency}s")

import pandas as pd
df_results = pd.DataFrame(results)
print(f"\nAvg latency per call: {df_results['latency_s'].mean():.2f}s")
print(f"Total match rate    : {(df_results['match'] == '✅').mean() * 100:.0f}%")   # FIXED: use '✅'

AVG_LATENCY = round(df_results['latency_s'].mean(), 2)
MATCH_RATE = round((df_results['match'] == '✅').mean() * 100, 0)

Batch inference on 5 reviews...

review_id=9 | rating=1 | actual=negative  | llm=positive: | ❌ | 0.72s
review_id=24 | rating=2 | actual=positive  | llm=positive: | ❌ | 0.93s
review_id=5 | rating=3 | actual=neutral   | llm=positive: | ❌ | 0.8s
review_id=1 | rating=4 | actual=negative  | llm=positive: | ❌ | 0.95s
review_id=3 | rating=5 | actual=positive  | llm=positive  | ✅ | 1.08s

Avg latency per call: 0.90s
Total match rate    : 20%


In [14]:
# ── T5b: FULL STATS SUMMARY FOR NRA ─────────────────────────────────────────
# Goal: Compute all numbers needed for the NRA insight from printed output
# Rule: All NRA numbers MUST come from this cell's printed output

print("=== ReviewPulse Summary Stats (for NRA) ===")
print(f"Total reviews       : {len(df_raw)}")
print(f"High-rating reviews : {df_raw['high_rating'].sum()} ({df_raw['high_rating'].mean()*100:.1f}%)")
print(f"Low-rating reviews  : {(df_raw['high_rating']==0).sum()} ({(df_raw['high_rating']==0).mean()*100:.1f}%)")
print()

# Per-category high-rating rate
cat_stats = df_raw.groupby('product_category').agg(
    n=('review_id','count'),
    high_rating_pct=('high_rating', lambda x: round(x.mean()*100, 1)),
    avg_rating=('rating', lambda x: round(x.mean(), 2))
).sort_values('high_rating_pct', ascending=False)
print("Per-category stats:")
print(cat_stats.to_string())

print()
print(f"=== Ollama Inference Stats ===")
print(f"Avg latency per call: {AVG_LATENCY}s")
print(f"LLM match rate      : {MATCH_RATE}%")


=== ReviewPulse Summary Stats (for NRA) ===
Total reviews       : 600
High-rating reviews : 316 (52.7%)
Low-rating reviews  : 284 (47.3%)

Per-category stats:
                    n  high_rating_pct  avg_rating
product_category                                  
Laptop            129             55.8        3.42
Headphones        121             54.5        3.35
Tablet            124             53.2        3.56
Smartwatch        118             52.5        3.37
Smartphone        108             46.3        3.21

=== Ollama Inference Stats ===
Avg latency per call: 0.9s
LLM match rate      : 20.0%


In [20]:
# ── T5c: NRA INSIGHT ─────────────────────────────────────────────────────────
# Goal: Write ONE NRA (Number + Reason + Action) insight from batch inference.
# Numbers must come from PRINTED output above – no estimation.

print("""
=== NRA INSIGHT: Ollama Local LLM on ReviewPulse India ===

**Number:** The LLM achieved a match rate of only 20% (1 out of 5 reviews) with an average latency of 0.90 seconds per inference. The dataset has a high‑rating rate of 52.7% (316/600) and a per‑category high‑rating range from 46.3% (Smartphone) to 55.8% (Laptop).

**Reason:** The low match rate is expected because the prompt instructs the model to classify based solely on the numeric rating (1‑2=negative, 3=neutral, 4‑5=positive), but the actual `sentiment` column does not strictly follow that mapping – for example, a review with rating 4 can be labeled "negative" (as seen in review_id=1). Additionally, TinyLlama (1.1B) may produce verbosity or misinterpret the instruction, returning "positive" for ratings that should be negative, leading to frequent mismatches. The latency (0.9s) is acceptable for batch workloads but too high for real‑time applications.

**Action:** For production deployment, replace the rule‑based classification prompt with a **zero‑shot sentiment analysis using the actual review text** (instead of just rating) and switch to a larger model like `phi3:mini` (3.8B) or use Groq (cloud) for latency‑critical tasks. This should improve match rate to at least 80% while keeping latency under 0.5s with Groq. If staying local, implement a **hybrid pipeline**: use the rating rule as a fast fallback and invoke the LLM only for reviews where rating and sentiment are misaligned (e.g., rating 4 but sentiment negative).
""")


=== NRA INSIGHT: Ollama Local LLM on ReviewPulse India ===

**Number:** The LLM achieved a match rate of only 20% (1 out of 5 reviews) with an average latency of 0.90 seconds per inference. The dataset has a high‑rating rate of 52.7% (316/600) and a per‑category high‑rating range from 46.3% (Smartphone) to 55.8% (Laptop).

**Reason:** The low match rate is expected because the prompt instructs the model to classify based solely on the numeric rating (1‑2=negative, 3=neutral, 4‑5=positive), but the actual `sentiment` column does not strictly follow that mapping – for example, a review with rating 4 can be labeled "negative" (as seen in review_id=1). Additionally, TinyLlama (1.1B) may produce verbosity or misinterpret the instruction, returning "positive" for ratings that should be negative, leading to frequent mismatches. The latency (0.9s) is acceptable for batch workloads but too high for real‑time applications.

**Action:** For production deployment, replace the rule‑based classific

---
### ★ Bonus: Groq vs Ollama Latency Comparison (10★)

In [19]:
# ── BONUS: GROQ vs OLLAMA LATENCY COMPARISON (using Colab Secrets) ──────────
# Goal: Compare inference latency between Groq (cloud) and local Ollama
#       using REST APIs. Securely reads the Groq API key from Colab secrets.

import requests
import time
import pandas as pd

# --- Securely get Groq API key from Colab secrets ---
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    print("✅ Groq API key loaded from Colab secrets.")
except (ImportError, userdata.SecretNotFoundError, userdata.NotebookAccessError) as e:
    print(f"⚠️  Could not load Groq API key: {e}")
    GROQ_API_KEY = None

# --- Shared prompt ---
PROMPT_TEMPLATE = (
    "A {product} has an average rating of 3.8/5 with 40% negative reviews. "
    "In exactly one sentence, state the single most important improvement action."
)

products_bench = ['Smartphone', 'Laptop', 'Headphones']
results_bench = []

for product in products_bench:
    prompt = PROMPT_TEMPLATE.format(product=product)

    # --- Ollama (local) ---
    ollama_payload = {
        "model": "tinyllama",
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.0, "num_predict": 80}
    }
    t0 = time.time()
    resp_ollama = requests.post('http://localhost:11434/api/generate', json=ollama_payload, timeout=60)
    latency_ollama = round(time.time() - t0, 3)
    if resp_ollama.status_code == 200:
        ollama_response = resp_ollama.json().get('response', '')
    else:
        ollama_response = f"Error {resp_ollama.status_code}"

    # --- Groq (cloud) – only if key is available ---
    if GROQ_API_KEY:
        groq_headers = {
            "Authorization": f"Bearer {GROQ_API_KEY}",
            "Content-Type": "application/json"
        }
        groq_payload = {
            "model": "llama-3.1-8b-instant",
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.0,
            "max_tokens": 80
        }
        t0 = time.time()
        resp_groq = requests.post('https://api.groq.com/openai/v1/chat/completions',
                                  json=groq_payload, headers=groq_headers, timeout=60)
        latency_groq = round(time.time() - t0, 3)
        if resp_groq.status_code == 200:
            groq_response = resp_groq.json()['choices'][0]['message']['content']
        else:
            groq_response = f"Error {resp_groq.status_code}"
    else:
        latency_groq = None
        groq_response = "(API key not set)"

    results_bench.append({
        'Product': product,
        'Ollama_latency_s': latency_ollama,
        'Groq_latency_s': latency_groq if latency_groq is not None else None,
        'Faster': 'Groq' if (latency_groq and latency_groq < latency_ollama) else 'Ollama'
    })
    print(f"{product}: Ollama={latency_ollama}s | Groq={latency_groq}s")

# --- Display results ---
df_bench = pd.DataFrame(results_bench)
print("\n=== LATENCY COMPARISON TABLE ===")
print(df_bench.to_string(index=False))

if GROQ_API_KEY:
    avg_oll = df_bench['Ollama_latency_s'].mean()
    avg_groq = df_bench['Groq_latency_s'].dropna().mean()
    speedup = round(avg_oll / avg_groq, 1) if avg_groq else None
    print(f"\nAvg Ollama latency : {avg_oll:.3f}s")
    print(f"Avg Groq latency   : {avg_groq:.3f}s")
    if speedup:
        print(f"Groq speedup       : {speedup}x faster")
else:
    print("\n⚠️  Groq API key not available – skipping Groq comparison.")

✅ Groq API key loaded from Colab secrets.
Smartphone: Ollama=0.489s | Groq=0.332s
Laptop: Ollama=0.404s | Groq=0.277s
Headphones: Ollama=0.335s | Groq=0.255s

=== LATENCY COMPARISON TABLE ===
   Product  Ollama_latency_s  Groq_latency_s Faster
Smartphone             0.489           0.332   Groq
    Laptop             0.404           0.277   Groq
Headphones             0.335           0.255   Groq

Avg Ollama latency : 0.409s
Avg Groq latency   : 0.288s
Groq speedup       : 1.4x faster


---
## Section 4: Scoring Rubric

| Task | Sub-task | Points | What is checked |
|------|----------|--------|-----------------|
| **T1** | Install Ollama | 5 | Binary installs, version printed |
| **T1** | Start server subprocess | 5 | Popen used (not Popen blocking), PID printed |
| **T1** | Health check HTTP 200 | 5 | requests.get to port 11434, status checked |
| **T2** | Pull TinyLlama | 8 | Pull succeeds, no timeout |
| **T2** | Verify via API + size printed | 7 | /api/tags used, size in MB printed |
| **T3** | REST API POST + latency | 12 | stream=False, OLLAMA_LATENCY_S captured |
| **T3** | Response fields parsed | 8 | eval_count, total_duration, tokens/sec printed |
| **T4** | Ollama LLM via LangChain | 10 | Correct import path, invoke works |
| **T4** | LCEL chain + 3 products | 10 | Chain built with pipe operator, real stats used |
| **T5** | Batch inference on 5 reviews | 12 | Loop correct, match comparison printed |
| **T5** | NRA — Number | 3 | From printed output |
| **T5** | NRA — Reason | 2 | Causal mechanism, not circular |
| **T5** | NRA — Action | 3 | Specific model/parameter, measurable target |
| **★** | Groq vs Ollama comparison | 10 | Both run, latency table printed, speedup computed |
| **Total** | | **90+10★** | |

### Automatic deductions
- Importing Ollama from wrong path → -5 (T4)
- NRA Number not from printed output → -3 (T5)
- `stream=True` in T3 (response not parseable as single JSON) → -5
- Server started with blocking Popen (no other cells can run) → -8

---

### Interview Answer (Month 10 Framing)
*"Ollama lets me package open-source LLMs as a local HTTP server, which I run on Colab's T4 GPU when my local machine doesn't have the VRAM. The key production advantage is that I can integrate it into LangChain with one import swap — `Ollama` in place of `ChatGroq` — so the same LCEL chain works whether I'm offline for sensitive client data or on cloud for speed. In a real deployment, I'd use Groq for latency-critical inference and local Ollama for data that can't leave the client's network."*

---

**GitHub commit:**
```
feat: Day175 - Ollama on Colab Local LLM Inference [pending]
```
Repo: `Month10-LangChain-MLflow-Portfolio`
